## IMPORT & SETUP


In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import os
from google.colab import drive

print(os.getcwd())
drive.mount("/content/drive")
df = pd.read_csv(
    "/content/drive/MyDrive/SE-dataset/cleaned_diseases_symptoms.csv"
)
print(f"Shape: {df.shape}")
print(f"Unique diseases: {df['diseases'].nunique()}")

/content
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (101723, 377)
Unique diseases: 773


## SEPARATING FEATURES AND TARGET


In [5]:
# Filter out diseases with fewer than 5 samples (minimum for stratified split)
counts = df['diseases'].value_counts()
df = df[df['diseases'].isin(counts[counts >= 5].index)]

print(f"Shape after filter: {df.shape}")
print(f"Unique diseases: {df['diseases'].nunique()}")

X = df.drop(columns=["diseases"])
y = df["diseases"]

Shape after filter: (101485, 377)
Unique diseases: 658


## ENCODING TARGET LABELS


In [6]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
os.makedirs('../models', exist_ok = True)
joblib.dump(le, '../models/label_encoder.pk')

['../models/label_encoder.pk']

## Train / val / test split (60/20/20 stratified)


In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.4, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Train: (60891, 376) | Val: (20297, 376) | Test: (20297, 376)


## TRAIN RANDOM FOREST


In [10]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,  # unconstrained first run
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=1,  # parallelism
)

print("Training...")
rf.fit(X_train, y_train)
print("Done")

Training...
Done


## EVALUATE

In [12]:
for name, X_set, y_set in [("Train", X_train, y_train),
                           ("Val", X_val, y_val),
                           ("Test", X_test, y_test)]:
  pred = rf.predict(X_set)
  acc = accuracy_score(y_set, pred)
  f1 = f1_score(y_set, pred, average = 'macro', zero_division = 0)
  print(f"{name} - Accuracy: {acc} | Macro-F1: {f1}")

Train - Accuracy: 0.9069977500780082 | Macro-F1: 0.859241695645873
Val - Accuracy: 0.8279056018130758 | Macro-F1: 0.7476191949381987
Test - Accuracy: 0.8266246243287185 | Macro-F1: 0.7501031702802243


## SAVING MODEL

In [13]:
joblib.dump(rf, '../models/random_forest.pk1')
print("Model saved")

Model saved
